<a href="https://colab.research.google.com/gist/kimchi-chisung/596f1e9626d1a86746b32838c3323247/a10_code.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# CEE6501 — Final Project: input file generation


---

## Define functions

In [1]:
import json

def build_chaotianmen_elements(
    E,
    A_bottom,
    I_bottom,
    J_bottom,
    A_top,
    I_top,
    J_top,
    A_top_vertical,
    A_bottom_vertical,
    A_top_diagonal,
    A_bottom_diagonal,
    A_cable,
):
    elements = {}
    eid = 1

    def add_element(elem_type, n1, n2, E, A=None, I=None, J=None):
        nonlocal eid
        elem = {
            "type": elem_type,
            "nodes": [n1, n2],
            "E": E,
        }

        if elem_type == "3D_frame":
            elem["A"] = A
            elem["I"] = I
            elem["J"] = J
        elif elem_type in ["3D_truss", "3D_cable"]:
            elem["A"] = A
        else:
            raise ValueError(f"Unknown element type: {elem_type}")

        elements[str(eid)] = elem
        eid += 1

    # 1) Bottom chord
    for n in range(1, 71):
        add_element("3D_frame", n, n + 1, E, A=A_bottom, I=I_bottom, J=J_bottom)

    bottom_chord = [
        (10, 143), (143, 144), (144, 145), (145, 146),
        (146, 147), (147, 148), (148, 149), (149, 18), (18, 150), (184, 54),
        (54, 185), (185, 186), (186, 187), (187, 188),
        (188, 189), (189, 190), (190, 191), (191, 62),
    ]
    for i, j in bottom_chord:
        add_element("3D_frame", i, j, E, A=A_bottom, I=I_bottom, J=J_bottom)

    for n in range(150, 185):
        add_element("3D_frame", n, n + 1, E, A=A_bottom, I=I_bottom, J=J_bottom)

    # 2) Top chord / arch
    for n in range(72, 142):
        add_element("3D_frame", n, n + 1, E, A=A_top, I=I_top, J=J_top)

    # 3) Top verticals
    top_vertical = [
        (1, 15, 71),
        (16, 19, 71),
        (54, 57, 71),
        (58, 72, 71),
        (90, 125, 60),
    ]
    for start, stop, offset in top_vertical:
        for n in range(start, stop):
            add_element("3D_truss", n, n + offset, E, A=A_top_vertical)

    top_vertical2 = [
        (86, 192), (192, 15), (128, 193), (193, 57),
    ]
    for i, j in top_vertical2:
        add_element("3D_truss", i, j, E, A=A_top_vertical)

    # 4) Bottom verticals
    bottom_vertical = [
        (11, 15, 132),
        (16, 18, 132),
        (55, 57, 130),
        (58, 62, 130),
    ]
    for start, stop, offset in bottom_vertical:
        for n in range(start, stop):
            add_element("3D_truss", n, n + offset, E, A=A_bottom_vertical)

    bottom_vertical2 = [
        (15, 194), (194, 147), (57, 195), (195, 187),
    ]
    for i, j in bottom_vertical2:
        add_element("3D_truss", i, j, E, A=A_bottom_vertical)

    # 5) Top diagonals
    top_diagonal = [
        (1, 14, 72),
        (17, 19, 70),
        (89, 107, 61),
        (108, 126, 59),
        (54, 56, 72),
        (59, 72, 70),
    ]
    for start, stop, offset in top_diagonal:
        for n in range(start, stop):
            add_element("3D_truss", n, n + offset, E, A=A_top_diagonal)

    top_diagonal2 = [
        (85, 192), (87, 192), (14, 192), (16, 192),
        (127, 193), (56, 193), (58, 193), (129, 193),
    ]
    for i, j in top_diagonal2:
        add_element("3D_truss", i, j, E, A=A_top_diagonal)

    # 6) Bottom diagonals
    bottom_diagonal = [
        (194, 14), (194, 16), (194, 146), (194, 148),
        (195, 56), (195, 58), (195, 186), (195, 188),
    ]
    for i, j in bottom_diagonal:
        add_element("3D_truss", i, j, E, A=A_bottom_diagonal)

    # 7) Cables
    cable = [
        (19, 54, 131),
    ]
    for start, stop, offset in cable:
        for n in range(start, stop):
            add_element("3D_cable", n, n + offset, E, A=A_cable)

    return elements


def duplicate_nodes_y_symmetric(nodes, width, base_node_end=195):
    """
    Original nodes:
        1 ~ base_node_end
    Copied nodes:
        base_node_end+1 ~ 2*base_node_end

    Original plane -> y = -width/2
    Copied plane   -> y = +width/2
    """
    half = width / 2.0
    new_nodes = {}
    node_map = {}

    for nid in range(1, base_node_end + 1):
        x, _, z = nodes[str(nid)]
        new_nodes[str(nid)] = [x, -half, z]

    for nid in range(1, base_node_end + 1):
        x, _, z = nodes[str(nid)]
        new_id = nid + base_node_end
        new_nodes[str(new_id)] = [x, half, z]
        node_map[nid] = new_id

    return new_nodes, node_map


def duplicate_elements_with_offset(elements, node_map, start_eid=None):
    existing_eids = sorted(int(k) for k in elements.keys())
    if start_eid is None:
        start_eid = max(existing_eids) + 1

    new_elements = {}
    eid = start_eid

    for old_eid in existing_eids:
        elem = elements[str(old_eid)]
        i, j = elem["nodes"]

        new_elem = dict(elem)
        new_elem["nodes"] = [node_map[i], node_map[j]]

        new_elements[str(eid)] = new_elem
        eid += 1

    return new_elements, eid

def add_transverse_mixed_members(
    node_map,
    start_eid,
    E_frame,
    A_frame,
    I_frame,
    J_frame,
    E_truss,
    A_truss,
    frame_node_range=(1, 71),
):
    elements = {}
    eid = start_eid

    n_start, n_end = frame_node_range

    for old_node, new_node in node_map.items():
        if n_start <= old_node <= n_end:
            elements[str(eid)] = {
                "type": "3D_frame",
                "nodes": [old_node, new_node],
                "E": E_frame,
                "A": A_frame,
                "I": I_frame,
                "J": J_frame,
            }
        else:
            elements[str(eid)] = {
                "type": "3D_truss",
                "nodes": [old_node, new_node],
                "E": E_truss,
                "A": A_truss,
            }
        eid += 1

    return elements, eid

def write_model_pretty_compact(model, filepath):
    compact_dict_keys = {
        "nodes",
        "elements",
        "supports",
        "nodal_loads",
        "prescribed_displacements",
        "member_loads",
        "temperature_loads",
        "fabrication_errors",
        "releases",
    }

    with open(filepath, "w", encoding="utf-8") as f:
        f.write("{\n")

        keys = list(model.keys())
        for idx, key in enumerate(keys):
            value = model[key]
            f.write(f'  "{key}": ')

            if key in compact_dict_keys and isinstance(value, dict):
                f.write("{\n")
                subkeys = list(value.keys())
                for j, skey in enumerate(subkeys):
                    comma = "," if j < len(subkeys) - 1 else ""
                    val_str = json.dumps(value[skey], ensure_ascii=False)
                    f.write(f'    "{skey}": {val_str}{comma}\n')
                f.write("  }")
            else:
                txt = json.dumps(value, ensure_ascii=False, indent=2)
                txt = "\n".join("  " + line for line in txt.splitlines())
                f.write(txt[2:])

            if idx < len(keys) - 1:
                f.write(",")
            f.write("\n")

        f.write("}\n")

def build_chaotianmen_two_plane_model(
    nodes_base,
    supports_final,
    nodal_loads_final,
    E,
    A_bottom,
    I_bottom,
    J_bottom,
    A_top,
    I_top,
    J_top,
    A_top_vertical,
    A_bottom_vertical,
    A_top_diagonal,
    A_bottom_diagonal,
    A_cable,
    width_y=29.0,
    base_node_end=195,
    E_trans=200000000.0,
    A_trans=0.05,
):
    elements_base = build_chaotianmen_elements(
        E=E,
        A_bottom=A_bottom,
        I_bottom=I_bottom,
        J_bottom=J_bottom,
        A_top=A_top,
        I_top=I_top,
        J_top=J_top,
        A_top_vertical=A_top_vertical,
        A_bottom_vertical=A_bottom_vertical,
        A_top_diagonal=A_top_diagonal,
        A_bottom_diagonal=A_bottom_diagonal,
        A_cable=A_cable,
    )

    nodes_3d, node_map = duplicate_nodes_y_symmetric(
        nodes_base,
        width=width_y,
        base_node_end=base_node_end,
    )

    elements_copy, next_eid = duplicate_elements_with_offset(
        elements_base,
        node_map,
    )

    transverse_elements, next_eid = add_transverse_mixed_members(
        node_map=node_map,
        start_eid=next_eid,
        E_frame=E_TRANS_FRAME,
        A_frame=A_TRANS_FRAME,
        I_frame=I_TRANS_FRAME,
        J_frame=J_TRANS_FRAME,
        E_truss=E_TRANS_TRUSS,
        A_truss=A_TRANS_TRUSS,
        frame_node_range=(1, 71),
    )

    elements_3d = {}
    elements_3d.update(elements_base)
    elements_3d.update(elements_copy)
    elements_3d.update(transverse_elements)

    return {
        "nodes": nodes_3d,
        "elements": elements_3d,
        "supports": supports_final,
        "nodal_loads": nodal_loads_final,
        "node_map": node_map,
    }


---

## Define Nodes & Supports & Nodal Loads

In [6]:
nodes = {
    "1": [0.0, 0.0, 0.0],
    "2": [13.314286, 0.0, 0.0],
    "3": [26.628571, 0.0, 0.0],
    "4": [39.942857, 0.0, 0.0],
    "5": [53.257143, 0.0, 0.0],
    "6": [66.571429, 0.0, 0.0],
    "7": [79.885714, 0.0, 0.0],
    "8": [93.2, 0.0, 0.0],
    "9": [106.51429, 0.0, 0.0],
    "10": [119.82857, 0.0, 0.0],
    "11": [133.14286, 0.0, 0.0],
    "12": [146.45714, 0.0, 0.0],
    "13": [159.77143, 0.0, 0.0],
    "14": [173.08571, 0.0, 0.0],
    "15": [186.4, 0.0, 0.0],
    "16": [199.71429, 0.0, 0.0],
    "17": [213.02857, 0.0, 0.0],
    "18": [226.34286, 0.0, 0.0],
    "19": [239.65714, 0.0, 0.0],
    "20": [252.97143, 0.0, 0.0],
    "21": [266.28571, 0.0, 0.0],
    "22": [279.6, 0.0, 0.0],
    "23": [292.91429, 0.0, 0.0],
    "24": [306.22857, 0.0, 0.0],
    "25": [319.54286, 0.0, 0.0],
    "26": [332.85714, 0.0, 0.0],
    "27": [346.17143, 0.0, 0.0],
    "28": [359.48571, 0.0, 0.0],
    "29": [372.8, 0.0, 0.0],
    "30": [386.11429, 0.0, 0.0],
    "31": [399.42857, 0.0, 0.0],
    "32": [412.74286, 0.0, 0.0],
    "33": [426.05714, 0.0, 0.0],
    "34": [439.37143, 0.0, 0.0],
    "35": [452.68571, 0.0, 0.0],
    "36": [466.0, 0.0, 0.0],
    "37": [479.31429, 0.0, 0.0],
    "38": [492.62857, 0.0, 0.0],
    "39": [505.94286, 0.0, 0.0],
    "40": [519.25714, 0.0, 0.0],
    "41": [532.57143, 0.0, 0.0],
    "42": [545.88571, 0.0, 0.0],
    "43": [559.2, 0.0, 0.0],
    "44": [572.51429, 0.0, 0.0],
    "45": [585.82857, 0.0, 0.0],
    "46": [599.14286, 0.0, 0.0],
    "47": [612.45714, 0.0, 0.0],
    "48": [625.77143, 0.0, 0.0],
    "49": [639.08571, 0.0, 0.0],
    "50": [652.4, 0.0, 0.0],
    "51": [665.71429, 0.0, 0.0],
    "52": [679.02857, 0.0, 0.0],
    "53": [692.34286, 0.0, 0.0],
    "54": [705.65714, 0.0, 0.0],
    "55": [718.97143, 0.0, 0.0],
    "56": [732.28571, 0.0, 0.0],
    "57": [745.6, 0.0, 0.0],
    "58": [758.91429, 0.0, 0.0],
    "59": [772.22857, 0.0, 0.0],
    "60": [785.54286, 0.0, 0.0],
    "61": [798.85714, 0.0, 0.0],
    "62": [812.17143, 0.0, 0.0],
    "63": [825.48571, 0.0, 0.0],
    "64": [838.8, 0.0, 0.0],
    "65": [852.11429, 0.0, 0.0],
    "66": [865.42857, 0.0, 0.0],
    "67": [878.74286, 0.0, 0.0],
    "68": [892.05714, 0.0, 0.0],
    "69": [905.37143, 0.0, 0.0],
    "70": [918.68571, 0.0, 0.0],
    "71": [932.0, 0.0, 0.0],

    "72": [0.0, 0.0, 12.86785],
    "73": [13.314286, 0.0, 12.86785],
    "74": [26.628571, 0.0, 12.86785],
    "75": [39.942857, 0.0, 12.86785],
    "76": [53.257143, 0.0, 13.786982],
    "77": [66.571429, 0.0, 14.706114],
    "78": [79.885714, 0.0, 15.625247],
    "79": [93.2, 0.0, 16.544379],
    "80": [106.51429, 0.0, 17.463511],
    "81": [119.82857, 0.0, 19.301775],
    "82": [133.14286, 0.0, 22.059172],
    "83": [146.45714, 0.0, 23.897436],
    "84": [159.77143, 0.0, 26.654832],
    "85": [173.08571, 0.0, 30.331361],
    "86": [186.4, 0.0, 34.00789],
    "87": [199.71429, 0.0, 37.684418],
    "88": [213.02857, 0.0, 42.280079],
    "89": [226.34286, 0.0, 46.87574],
    "90": [239.65714, 0.0, 52.390533],
    "91": [252.97143, 0.0, 56.986193],
    "92": [266.28571, 0.0, 62.500986],
    "93": [279.6, 0.0, 67.096647],
    "94": [292.91429, 0.0, 71.692308],
    "95": [306.22857, 0.0, 76.287968],
    "96": [319.54286, 0.0, 79.964497],
    "97": [332.85714, 0.0, 83.641026],
    "98": [346.17143, 0.0, 86.398422],
    "99": [359.48571, 0.0, 89.155819],
    "100": [372.8, 0.0, 91.913215],
    "101": [386.11429, 0.0, 93.751479],
    "102": [399.42857, 0.0, 95.589744],
    "103": [412.74286, 0.0, 97.428008],
    "104": [426.05714, 0.0, 98.34714],
    "105": [439.37143, 0.0, 99.266272],
    "106": [452.68571, 0.0, 100.1854],
    "107": [466.0, 0.0, 100.1854],
    "108": [479.31429, 0.0, 100.1854],
    "109": [492.62857, 0.0, 99.266272],
    "110": [505.94286, 0.0, 98.34714],
    "111": [519.25714, 0.0, 97.428008],
    "112": [532.57143, 0.0, 95.589744],
    "113": [545.88571, 0.0, 93.751479],
    "114": [559.2, 0.0, 91.913215],
    "115": [572.51429, 0.0, 89.155819],
    "116": [585.82857, 0.0, 86.398422],
    "117": [599.14286, 0.0, 83.641026],
    "118": [612.45714, 0.0, 79.964497],
    "119": [625.77143, 0.0, 76.287968],
    "120": [639.08571, 0.0, 71.692308],
    "121": [652.4, 0.0, 67.096647],
    "122": [665.71429, 0.0, 62.500986],
    "123": [679.02857, 0.0, 56.986193],
    "124": [692.34286, 0.0, 52.390533],
    "125": [705.65714, 0.0, 46.87574],
    "126": [718.97143, 0.0, 42.280079],
    "127": [732.28571, 0.0, 37.684418],
    "128": [745.6, 0.0, 34.00789],
    "129": [758.91429, 0.0, 30.331361],
    "130": [772.22857, 0.0, 26.654832],
    "131": [785.54286, 0.0, 23.897436],
    "132": [798.85714, 0.0, 22.059172],
    "133": [812.17143, 0.0, 19.301775],
    "134": [825.48571, 0.0, 17.463511],
    "135": [838.8, 0.0, 16.544379],
    "136": [852.11429, 0.0, 15.625247],
    "137": [865.42857, 0.0, 14.706114],
    "138": [878.74286, 0.0, 13.786982],
    "139": [892.05714, 0.0, 12.86785],
    "140": [905.37143, 0.0, 12.86785],
    "141": [918.68571, 0.0, 12.86785],
    "142": [932.0, 0.0, 12.86785],

    "143": [133.14286, 0.0, -5.5147929],
    "144": [146.45714, 0.0, -12.86785],
    "145": [159.77143, 0.0, -20.220907],
    "146": [173.08571, 0.0, -29.412229],
    "147": [186.4, 0.0, -39.522682],
    "148": [199.71429, 0.0, -25.7357],
    "149": [213.02857, 0.0, -11.948718],

    "150": [239.65714, 0.0, 12.86785],
    "151": [252.97143, 0.0, 23.897436],
    "152": [266.28571, 0.0, 33.088757],
    "153": [279.6, 0.0, 40.441815],
    "154": [292.91429, 0.0, 47.794872],
    "155": [306.22857, 0.0, 53.309665],
    "156": [319.54286, 0.0, 58.824458],
    "157": [332.85714, 0.0, 64.33925],
    "158": [346.17143, 0.0, 68.934911],
    "159": [359.48571, 0.0, 73.530572],
    "160": [372.8, 0.0, 76.287968],
    "161": [386.11429, 0.0, 79.964497],
    "162": [399.42857, 0.0, 81.802761],
    "163": [412.74286, 0.0, 84.560158],
    "164": [426.05714, 0.0, 86.398422],
    "165": [439.37143, 0.0, 87.317554],
    "166": [452.68571, 0.0, 88.236686],
    "167": [466.0, 0.0, 88.236686],
    "168": [479.31429, 0.0, 88.236686],
    "169": [492.62857, 0.0, 87.317554],
    "170": [505.94286, 0.0, 86.398422],
    "171": [519.25714, 0.0, 84.560158],
    "172": [532.57143, 0.0, 81.802761],
    "173": [545.88571, 0.0, 79.964497],
    "174": [559.2, 0.0, 76.287968],
    "175": [572.51429, 0.0, 73.530572],
    "176": [585.82857, 0.0, 68.934911],
    "177": [599.14286, 0.0, 64.33925],
    "178": [612.45714, 0.0, 58.824458],
    "179": [625.77143, 0.0, 53.309665],
    "180": [639.08571, 0.0, 47.794872],
    "181": [652.4, 0.0, 40.441815],
    "182": [665.71429, 0.0, 33.088757],
    "183": [679.02857, 0.0, 23.897436],
    "184": [692.34286, 0.0, 12.86785],

    "185": [718.97143, 0.0, -11.94872],
    "186": [732.28571, 0.0, -25.7357],
    "187": [745.6, 0.0, -39.52268],
    "188": [758.91429, 0.0, -29.41223],
    "189": [772.22857, 0.0, -20.22091],
    "190": [785.54286, 0.0, -12.86785],
    "191": [798.85714, 0.0, -5.514793],
    "192": [186.4, 0.0, 15.62547],
    "193": [745.6, 0.0, 15.62547],
    "194": [186.4, 0.0, -14.70611],
    "195": [745.6, 0.0, -14.70611]
}

supports = {
    "1":   ["ux", "uy", "uz", "rx", "ry", "rz"],
    "71":  ["ux", "uy", "uz", "rx", "ry", "rz"],
    "147": ["ux", "uy", "uz", "rx", "ry", "rz"],
    "187": ["ux", "uy", "uz", "rx", "ry", "rz"],
    "196": ["ux", "uy", "uz", "rx", "ry", "rz"],
    "266": ["ux", "uy", "uz", "rx", "ry", "rz"],
    "342": ["ux", "uy", "uz", "rx", "ry", "rz"],
    "382": ["ux", "uy", "uz", "rx", "ry", "rz"],

    "192": ["ux", "uy", "uz", "rx", "ry", "rz"],
    "193": ["ux", "uy", "uz", "rx", "ry", "rz"],
    "194": ["ux", "uy", "uz", "rx", "ry", "rz"],
    "195": ["ux", "uy", "uz", "rx", "ry", "rz"],
    "387": ["ux", "uy", "uz", "rx", "ry", "rz"],
    "388": ["ux", "uy", "uz", "rx", "ry", "rz"],
    "389": ["ux", "uy", "uz", "rx", "ry", "rz"],
    "390": ["ux", "uy", "uz", "rx", "ry", "rz"],
}

nodal_loads = {
    "31": [0.0, 0.0, -200.0, 0.0, 0.0, 0.0],
    "32": [0.0, 0.0, -350.0, 0.0, 0.0, 0.0],
    "33": [0.0, 0.0, -500.0, 0.0, 0.0, 0.0],
    "34": [0.0, 0.0, -650.0, 0.0, 0.0, 0.0],
    "35": [0.0, 0.0, -800.0, 0.0, 0.0, 0.0],
    "36": [0.0, 0.0, -900.0, 0.0, 0.0, 0.0],
    "37": [0.0, 0.0, -800.0, 0.0, 0.0, 0.0],
    "38": [0.0, 0.0, -650.0, 0.0, 0.0, 0.0],
    "39": [0.0, 0.0, -500.0, 0.0, 0.0, 0.0],
    "40": [0.0, 0.0, -350.0, 0.0, 0.0, 0.0],
    "41": [0.0, 0.0, -200.0, 0.0, 0.0, 0.0],
}

In [7]:
nodal_loads = {}

---

## Generate an input JSON file

In [11]:
# ------------------------------------------------------------
# parameters
# ------------------------------------------------------------
E = 200000000.0

A_bottom = 0.50
I_bottom = 0.45
J_bottom = 0.05

A_top = 0.40
I_top = 0.30
J_top = 0.04

A_top_vertical = 0.08
A_bottom_vertical = 0.08
A_top_diagonal = 0.06
A_bottom_diagonal = 0.06
A_cable = 0.01

WIDTH_Y = 29.0
BASE_NODE_END = 195
E_TRANS = 200000000.0
A_TRANS = 0.05

E_TRANS_FRAME = 200000000.0
A_TRANS_FRAME = 0.05
I_TRANS_FRAME = 0.002
J_TRANS_FRAME = 0.001

E_TRANS_TRUSS = 200000000.0
A_TRANS_TRUSS = 0.05

# ------------------------------------------------------------
# build two-plane model
# ------------------------------------------------------------
model_3d = build_chaotianmen_two_plane_model(
    nodes_base=nodes,
    supports_final=supports,
    nodal_loads_final=nodal_loads,
    E=E,
    A_bottom=A_bottom,
    I_bottom=I_bottom,
    J_bottom=J_bottom,
    A_top=A_top,
    I_top=I_top,
    J_top=J_top,
    A_top_vertical=A_top_vertical,
    A_bottom_vertical=A_bottom_vertical,
    A_top_diagonal=A_top_diagonal,
    A_bottom_diagonal=A_bottom_diagonal,
    A_cable=A_cable,
    width_y=WIDTH_Y,
    base_node_end=BASE_NODE_END,
    E_trans=E_TRANS,
    A_trans=A_TRANS,
)

# ------------------------------------------------------------
# final JSON model
# ------------------------------------------------------------
model = {
    "model_name": "chaotianmen_bridge",
    "nodes": model_3d["nodes"],
    "elements": model_3d["elements"],
    "supports": model_3d["supports"],
    "nodal_loads": model_3d["nodal_loads"],
    "prescribed_displacements": {},
    "member_loads": {},
    "temperature_loads": {},
    "fabrication_errors": {},
    "releases": {},
}

# ------------------------------------------------------------
# save
# ------------------------------------------------------------
output_path = "inputs/chaotianmen_bridge.json"

write_model_pretty_compact(
    model,
    output_path,
)

print(f"Updated: {output_path}")
print(f"Original nodes : 1 ~ {BASE_NODE_END}")
print(f"Copied nodes   : {BASE_NODE_END + 1} ~ {2 * BASE_NODE_END}")
print(f"Width in y     : {WIDTH_Y} m")
print(f"Total nodes    : {2 * BASE_NODE_END}")
print(f"Total elements : {len(model_3d['elements'])}")

Updated: inputs/chaotianmen_bridge.json
Original nodes : 1 ~ 195
Copied nodes   : 196 ~ 390
Width in y     : 29.0 m
Total nodes    : 390
Total elements : 993


In [9]:
def merge_nodal_loads(*load_dicts):
    merged = {}

    for d in load_dicts:
        for nid, load in d.items():
            key = str(nid)
            if key not in merged:
                merged[key] = [0.0, 0.0, 0.0, 0.0, 0.0, 0.0]

            for i in range(6):
                merged[key][i] += load[i]

    return merged

def merge_member_loads(*load_dicts):
    merged = {}

    for d in load_dicts:
        for eid, loads in d.items():
            key = str(eid)
            if key not in merged:
                merged[key] = {}

            for load_name, val in loads.items():
                merged[key][load_name] = merged[key].get(load_name, 0.0) + val

    return merged

def build_self_weight_member_loads_by_type(
    elements,
    unit_weight_map=None,
    scale_factor_map=None,
    load_key="wz",
    default_unit_weight=None,
):
    """
    Build self-weight member loads by element type.

    Parameters
    ----------
    elements : dict
        Element dictionary.
    unit_weight_map : dict or None
        Unit weight by element type in kN/m^3.
        Example:
            {
                "3D_frame": 77.0,
                "3D_truss": 77.0,
                "3D_cable": 0.0,
            }
    scale_factor_map : dict or None
        Optional multiplier by element type.
        Example:
            {
                "3D_frame": 1.0,
                "3D_truss": 1.0,
                "3D_cable": 0.5,
            }
    load_key : str
        Usually "wz" for gravity in global z downward direction.
    default_unit_weight : float or None
        Fallback unit weight if an element type is missing in unit_weight_map.
        If None, missing types are skipped.

    Returns
    -------
    member_loads : dict
        {eid: {load_key: value}}
    """
    if unit_weight_map is None:
        unit_weight_map = {
            "3D_frame": 77.0,
            "3D_truss": 77.0,
            "3D_cable": 77.0,
        }

    if scale_factor_map is None:
        scale_factor_map = {}

    member_loads = {}

    for eid, elem in elements.items():
        etype = elem.get("type", None)
        if etype is None:
            continue

        if "A" not in elem:
            continue

        if etype in unit_weight_map:
            gamma = unit_weight_map[etype]
        elif default_unit_weight is not None:
            gamma = default_unit_weight
        else:
            continue

        scale = scale_factor_map.get(etype, 1.0)

        # If gamma or scale is zero, skip
        if gamma == 0.0 or scale == 0.0:
            continue

        A = elem["A"]
        w = gamma * A * scale   # kN/m

        member_loads[str(eid)] = {
            load_key: -w
        }

    return member_loads

def build_traffic_nodal_loads(
    loaded_nodes,
    wheel_line_loads,
    direction="z",
):
    """
    Build nodal traffic loads from a list of nodal vertical loads.

    Parameters
    ----------
    loaded_nodes : list[int]
        Node IDs where traffic loads are applied
    wheel_line_loads : list[float]
        Downward nodal loads in kN
    direction : str
        'x', 'y', or 'z'

    Returns
    -------
    nodal_loads : dict
    """
    if len(loaded_nodes) != len(wheel_line_loads):
        raise ValueError("loaded_nodes and wheel_line_loads must have the same length")

    dir_map = {"x": 0, "y": 1, "z": 2}
    idx = dir_map[direction]

    nodal_loads = {}

    for nid, p in zip(loaded_nodes, wheel_line_loads):
        load = [0.0, 0.0, 0.0, 0.0, 0.0, 0.0]
        load[idx] = -abs(p)
        nodal_loads[str(nid)] = load

    return nodal_loads

def build_train_nodal_loads(
    rail_nodes,
    nodes,
    axle_positions,
    axle_loads,
):
    """
    Build nodal train loads by snapping each axle to the nearest rail node in x.

    Parameters
    ----------
    rail_nodes : list[int]
        Candidate rail-supporting nodes
    nodes : dict
        Node coordinates
    axle_positions : list[float]
        Axle x-positions (m)
    axle_loads : list[float]
        Axle loads (kN), positive input

    Returns
    -------
    nodal_loads : dict
    """
    if len(axle_positions) != len(axle_loads):
        raise ValueError("axle_positions and axle_loads must have the same length")

    x_map = {}
    for nid in rail_nodes:
        coord = nodes[str(nid)] if str(nid) in nodes else nodes[nid]
        x_map[nid] = coord[0]

    nodal_loads = {}

    for x_axle, p_axle in zip(axle_positions, axle_loads):
        nearest_node = min(rail_nodes, key=lambda nid: abs(x_map[nid] - x_axle))

        key = str(nearest_node)
        if key not in nodal_loads:
            nodal_loads[key] = [0.0, 0.0, 0.0, 0.0, 0.0, 0.0]

        nodal_loads[key][2] -= abs(p_axle)

    return nodal_loads

def build_uniform_wind_member_loads(
    elements,
    element_ids=None,
    type_wind_map=None,
    direction="y",
):
    """
    Build uniform wind member loads by element type or selected element IDs.

    Parameters
    ----------
    elements : dict
        Element dictionary
    element_ids : list or None
        If given, apply only to these element IDs
    type_wind_map : dict or None
        Wind line load by type, in kN/m
        Example:
            {
                "3D_frame": 8.0,
                "3D_truss": 2.0,
                "3D_cable": 0.0,
            }
    direction : str
        'x', 'y', or 'z'

    Returns
    -------
    member_loads : dict
        {eid: {"wy": value}} or similar
    """
    if type_wind_map is None:
        type_wind_map = {
            "3D_frame": 0.0,
            "3D_truss": 0.0,
            "3D_cable": 0.0,
        }

    load_key = f"w{direction}"
    member_loads = {}

    selected = None if element_ids is None else {str(eid) for eid in element_ids}

    for eid, elem in elements.items():
        eid_str = str(eid)
        if selected is not None and eid_str not in selected:
            continue

        etype = elem["type"]
        w = type_wind_map.get(etype, 0.0)

        if abs(w) > 0.0:
            member_loads[eid_str] = {load_key: w}

    return member_loads

In [12]:
nodes = model_3d["nodes"]
elements = model_3d["elements"]
supports = model_3d["supports"]

# Self weight
member_loads_self_weight = build_self_weight_member_loads_by_type(
    elements,
    unit_weight_map={
        "3D_frame": 77.0,
        "3D_truss": 77.0,
        "3D_cable": 0.0,
    }
)

# Traffic load
road_nodes_upper = [31, 32, 33, 34, 35, 36, 37, 38, 39, 40, 41]
road_loads_moderate = [200, 350, 500, 650, 800, 900, 800, 650, 500, 350, 200]

nodal_loads_road = build_traffic_nodal_loads(
    loaded_nodes=road_nodes_upper,
    wheel_line_loads=road_loads_moderate,
    direction="z",
)

# High-speed train load
rail_nodes = [31, 32, 33, 34, 35, 36, 37, 38, 39, 40, 41]

axle_positions = [400, 413, 426, 439, 452, 465, 478, 491]  # m, example
axle_loads = [170, 170, 170, 170, 170, 170, 170, 170]      # kN, example

nodal_loads_train = build_train_nodal_loads(
    rail_nodes=rail_nodes,
    nodes=nodes,
    axle_positions=axle_positions,
    axle_loads=axle_loads,
)

# Wind load
member_loads_wind = build_uniform_wind_member_loads(
    elements=model_3d["elements"],
    type_wind_map={
        "3D_frame": 8.0,
        "3D_truss": 2.0,
        "3D_cable": 0.0,
    },
    direction="y",
)

# Merging same types of loads (member loading, node loading)
member_loads_total = merge_member_loads(
    member_loads_self_weight,
    member_loads_wind,
)

nodal_loads_total = merge_nodal_loads(
    nodal_loads_road,
    nodal_loads_train,
)

In [13]:
model = {
    "model_name": "chaotianmen_dead_road_train",
    "nodes": model_3d["nodes"],
    "elements": model_3d["elements"],
    "supports": model_3d["supports"],
    "nodal_loads": nodal_loads_total,
    "member_loads": member_loads_self_weight,
    "prescribed_displacements": {},
    "temperature_loads": {},
    "fabrication_errors": {},
    "releases": {},
}

write_model_pretty_compact(
    model,
    "inputs/chaotianmen_dead_road_train.json",
)

In [15]:
def build_uniform_temperature_loads(elements, deltaT, include_types=None):
    if include_types is None:
        include_types = {"3D_frame", "3D_truss", "3D_cable"}

    temperature_loads = {}
    for eid, elem in elements.items():
        if elem["type"] not in include_types:
            continue
        temperature_loads[str(eid)] = {"dT": float(deltaT)}
    return temperature_loads


T_ref = 20.0

temp_cases = {
    "winter_service": 8.0 - T_ref,
    "summer_service": 34.0 - T_ref,
    "extreme_cold": 0.0 - T_ref,
    "extreme_hot": 40.0 - T_ref,
}

for case_name, deltaT in temp_cases.items():
    temperature_loads = build_uniform_temperature_loads(
        elements=model_3d["elements"],
        deltaT=deltaT,
    )

    model = {
        "model_name": f"chaotianmen_dead_road_train_temp_uniform_{case_name}",
        "nodes": model_3d["nodes"],
        "elements": model_3d["elements"],
        "supports": model_3d["supports"],
        "nodal_loads": nodal_loads_total,
        "prescribed_displacements": {},
        "member_loads": member_loads_self_weight,
        "temperature_loads": temperature_loads,
        "fabrication_errors": {},
        "releases": {},
    }

    output_path = f"inputs/chaotianmen_dead_road_train_temp_uniform_{case_name}.json"

    write_model_pretty_compact(model, output_path)
    print(f"Updated: {output_path}")

Updated: inputs/chaotianmen_dead_road_train_temp_uniform_winter_service.json
Updated: inputs/chaotianmen_dead_road_train_temp_uniform_summer_service.json
Updated: inputs/chaotianmen_dead_road_train_temp_uniform_extreme_cold.json
Updated: inputs/chaotianmen_dead_road_train_temp_uniform_extreme_hot.json


In [14]:
# Fire

def build_local_fire_temperature_loads(element_ids, deltaT_fire):
    temperature_loads = {}
    for eid in element_ids:
        temperature_loads[str(eid)] = {"dT": float(deltaT_fire)}
    return temperature_loads


def find_elements_by_x_range(elements, nodes, x_min, x_max, element_types=None):
    selected = []

    for eid, elem in elements.items():
        if element_types is not None and elem["type"] not in element_types:
            continue

        i, j = elem["nodes"]
        xi = nodes[str(i)][0] if str(i) in nodes else nodes[i][0]
        xj = nodes[str(j)][0] if str(j) in nodes else nodes[j][0]

        x_mid = 0.5 * (xi + xj)

        if x_min <= x_mid <= x_max:
            selected.append(int(eid))

    return selected


# ------------------------------------------------------------
# local fire zone selection
# ------------------------------------------------------------
fire_zone_eids = find_elements_by_x_range(
    elements=model_3d["elements"],
    nodes=model_3d["nodes"],
    x_min=420.0,
    x_max=510.0,
    element_types={"3D_frame"},
)

temperature_loads = build_local_fire_temperature_loads(
    element_ids=fire_zone_eids,
    deltaT_fire=300.0,
)

# ------------------------------------------------------------
# final JSON model
# ------------------------------------------------------------
model = {
    "model_name": "chaotianmen_dead_road_train_temp_local_fire_midspan",
    "nodes": model_3d["nodes"],
    "elements": model_3d["elements"],
    "supports": model_3d["supports"],
    "nodal_loads": nodal_loads_total,
    "prescribed_displacements": {},
    "member_loads": member_loads_self_weight,
    "temperature_loads": temperature_loads,
    "fabrication_errors": {},
    "releases": {},
}

# ------------------------------------------------------------
# save
# ------------------------------------------------------------
output_path = "inputs/chaotianmen_dead_road_train_temp_local_fire_midspan.json"

write_model_pretty_compact(
    model,
    output_path,
)

print(f"Updated: {output_path}")
print("Fire-zone element count:", len(fire_zone_eids))
print("Sample fire-zone element IDs:", fire_zone_eids[:20])

Updated: inputs/chaotianmen_dead_road_train_temp_local_fire_midspan.json
Fire-zone element count: 43
Sample fire-zone element IDs: [33, 34, 35, 36, 37, 38, 103, 104, 105, 106, 107, 108, 156, 157, 158, 159, 160, 161, 432, 433]


In [15]:
from copy import deepcopy

def find_elements_by_x_range(elements, nodes, x_min, x_max, element_types=None):
    selected = []

    for eid, elem in elements.items():
        if element_types is not None and elem["type"] not in element_types:
            continue

        i, j = elem["nodes"]
        xi = nodes[str(i)][0] if str(i) in nodes else nodes[i][0]
        xj = nodes[str(j)][0] if str(j) in nodes else nodes[j][0]
        x_mid = 0.5 * (xi + xj)

        if x_min <= x_mid <= x_max:
            selected.append(int(eid))

    return selected


def find_elements_by_xz_range(elements, nodes, x_min, x_max, z_min, z_max, element_types=None):
    selected = []

    for eid, elem in elements.items():
        if element_types is not None and elem["type"] not in element_types:
            continue

        i, j = elem["nodes"]
        ci = nodes[str(i)] if str(i) in nodes else nodes[i]
        cj = nodes[str(j)] if str(j) in nodes else nodes[j]

        x_mid = 0.5 * (ci[0] + cj[0])
        z_mid = 0.5 * (ci[2] + cj[2])

        if x_min <= x_mid <= x_max and z_min <= z_mid <= z_max:
            selected.append(int(eid))

    return selected


def find_longest_elements(elements, nodes, top_n=10, element_types=None):
    rows = []

    for eid, elem in elements.items():
        if element_types is not None and elem["type"] not in element_types:
            continue

        i, j = elem["nodes"]
        xi, yi, zi = nodes[str(i)] if str(i) in nodes else nodes[i]
        xj, yj, zj = nodes[str(j)] if str(j) in nodes else nodes[j]

        L = ((xj - xi)**2 + (yj - yi)**2 + (zj - zi)**2) ** 0.5
        rows.append((int(eid), L))

    rows.sort(key=lambda x: x[1], reverse=True)
    return rows[:top_n]

In [21]:
# "fabrication_errors": {
#     "105": {"deltaL": 0.01}
# }

def build_fabrication_error_loads(element_ids, deltaL):
    """
    Apply the same fabrication length error to selected elements.

    Parameters
    ----------
    element_ids : iterable
        Element IDs
    deltaL : float
        Length error in meters (+ expansion / - shortening convention by your solver)

    Returns
    -------
    fabrication_errors : dict
        {eid: {"deltaL": deltaL}}
    """
    fabrication_errors = {}
    for eid in element_ids:
        fabrication_errors[str(eid)] = {"deltaL": float(deltaL)}
    return fabrication_errors


def build_fabrication_error_map(error_map):
    """
    Build fabrication error dictionary from explicit per-element values.

    Parameters
    ----------
    error_map : dict
        {eid: deltaL}

    Returns
    -------
    fabrication_errors : dict
    """
    fabrication_errors = {}
    for eid, deltaL in error_map.items():
        fabrication_errors[str(eid)] = {"deltaL": float(deltaL)}
    return fabrication_errors

# Arch crown
crown_eids = find_elements_by_x_range(
    elements=model_3d["elements"],
    nodes=model_3d["nodes"],
    x_min=430.0,
    x_max=500.0,
    element_types={"3D_frame"},
)

fabrication_errors = build_fabrication_error_loads(
    element_ids=crown_eids,
    deltaL=0.005,   # 5 mm
)

# Longest diagonals
long_diagonals = find_longest_elements(
    elements=model_3d["elements"],
    nodes=model_3d["nodes"],
    top_n=8,
    element_types={"3D_truss"},
)

long_diag_ids = [eid for eid, L in long_diagonals]

fabrication_errors = build_fabrication_error_loads(
    element_ids=long_diag_ids,
    deltaL=0.003,   # 3 mm
)

model = {
    "model_name": "chaotianmen_fabrication_error_crown_case1",
    "nodes": model_3d["nodes"],
    "elements": model_3d["elements"],
    "supports": model_3d["supports"],
    "nodal_loads": nodal_loads_total,
    "prescribed_displacements": {},
    "member_loads": member_loads_self_weight,
    "temperature_loads": {},
    "fabrication_errors": fabrication_errors,
    "releases": {},
}

write_model_pretty_compact(
    model,
    "inputs/chaotianmen_fabrication_error_crown_case1.json",
)

In [22]:
# Support settlement
# "prescribed_displacements": {
#     "1": {"uz": -0.01}
# }

def build_support_settlement(node_ids, dof, value):
    """
    Apply the same prescribed displacement to selected support nodes.

    Parameters
    ----------
    node_ids : iterable
        Node IDs
    dof : str
        One of 'ux', 'uy', 'uz', 'rx', 'ry', 'rz'
    value : float
        Prescribed displacement/rotation value

    Returns
    -------
    prescribed_displacements : dict
        {nid: {dof: value}}
    """
    prescribed_displacements = {}
    for nid in node_ids:
        prescribed_displacements[str(nid)] = {dof: float(value)}
    return prescribed_displacements


def build_support_settlement_map(settlement_map):
    """
    Build prescribed displacement dictionary from explicit map.

    Parameters
    ----------
    settlement_map : dict
        {
            node_id: {"uz": -0.01},
            ...
        }

    Returns
    -------
    prescribed_displacements : dict
    """
    prescribed_displacements = {}
    for nid, dofs in settlement_map.items():
        prescribed_displacements[str(nid)] = {
            k: float(v) for k, v in dofs.items()
        }
    return prescribed_displacements


def merge_prescribed_displacements(*disp_dicts):
    merged = {}
    for d in disp_dicts:
        for nid, dofs in d.items():
            if nid not in merged:
                merged[nid] = {}
            merged[nid].update(dofs)
    return merged


# Single support downward
prescribed_displacements = build_support_settlement(
    node_ids=[1],
    dof="uz",
    value=-0.01,   # 10 mm settlement
)

# One side only
prescribed_displacements = build_support_settlement_map({
    1: {"uz": -0.01},
    71: {"uz": -0.01},
    147: {"uz": -0.01},
    187: {"uz": -0.01},
})

# Opposite supports differential settlement
prescribed_displacements = build_support_settlement_map({
    1:   {"uz": -0.010},
    196: {"uz": -0.004},
    71:  {"uz": -0.010},
    266: {"uz": -0.004},
})

model = {
    "model_name": "chaotianmen_support_settlement_case1",
    "nodes": model_3d["nodes"],
    "elements": model_3d["elements"],
    "supports": model_3d["supports"],
    "nodal_loads": nodal_loads_total,
    "prescribed_displacements": prescribed_displacements,
    "member_loads": member_loads_self_weight,
    "temperature_loads": {},
    "fabrication_errors": {},
    "releases": {},
}

write_model_pretty_compact(
    model,
    "inputs/chaotianmen_support_settlement_case1.json",
)

In [23]:
# Damage / member removal

def remove_elements_from_model(elements, remove_element_ids):
    """
    Return a new elements dict with selected elements removed.
    """
    remove_set = {str(eid) for eid in remove_element_ids}
    new_elements = {
        str(eid): deepcopy(elem)
        for eid, elem in elements.items()
        if str(eid) not in remove_set
    }
    return new_elements


def reduce_element_area(elements, reduce_map, min_area_ratio=1e-6):
    """
    Return a new elements dict with reduced area on selected elements.
    Useful as a soft-damage alternative to full removal.

    Parameters
    ----------
    reduce_map : dict
        {eid: area_scale_factor}
        Example: {105: 0.1, 110: 0.01}
    """
    new_elements = deepcopy(elements)

    for eid, scale in reduce_map.items():
        key = str(eid)
        if key not in new_elements:
            continue
        if "A" in new_elements[key]:
            new_elements[key]["A"] *= max(float(scale), min_area_ratio)

    return new_elements


def reduce_element_stiffness_proxy(elements, reduce_ids, area_scale=1e-4, inertia_scale=1e-4):
    """
    Soft-damage model by reducing A/I/J for selected elements.
    """
    new_elements = deepcopy(elements)

    for eid in reduce_ids:
        key = str(eid)
        if key not in new_elements:
            continue

        if "A" in new_elements[key]:
            new_elements[key]["A"] *= area_scale
        if "I" in new_elements[key]:
            new_elements[key]["I"] *= inertia_scale
        if "J" in new_elements[key]:
            new_elements[key]["J"] *= inertia_scale

    return new_elements


# Central diagonal removal
central_diagonals = find_elements_by_x_range(
    elements=model_3d["elements"],
    nodes=model_3d["nodes"],
    x_min=430.0,
    x_max=500.0,
    element_types={"3D_truss"},
)

elements_damaged = remove_elements_from_model(
    elements=model_3d["elements"],
    remove_element_ids=central_diagonals[:2],
)

# Transverse bracing removal
transverse_eids = find_longest_elements(
    elements=model_3d["elements"],
    nodes=model_3d["nodes"],
    top_n=20,
    element_types={"3D_truss"},
)

remove_ids = [eid for eid, L in transverse_eids[:4]]

elements_damaged = remove_elements_from_model(
    elements=model_3d["elements"],
    remove_element_ids=remove_ids,
)

model = {
    "model_name": "chaotianmen_damage_remove_diagonal_case1",
    "nodes": model_3d["nodes"],
    "elements": elements_damaged,
    "supports": model_3d["supports"],
    "nodal_loads": nodal_loads_total,
    "prescribed_displacements": {},
    "member_loads": member_loads_self_weight,
    "temperature_loads": {},
    "fabrication_errors": {},
    "releases": {},
}

write_model_pretty_compact(
    model,
    "inputs/chaotianmen_damage_remove_diagonal_case1.json",
)

In [24]:
# Moment release / fixity
# "releases": {
#     "1": ["j_rz"],
#     "2": ["i_rz"]
# }

def build_element_releases(element_ids, release_labels):
    """
    Apply the same release labels to selected elements.

    Parameters
    ----------
    element_ids : iterable
        Element IDs
    release_labels : list[str]
        Example: ["i_rz"], ["j_rz"], ["i_ry", "j_ry"]

    Returns
    -------
    releases : dict
        {eid: [release_labels...]}
    """
    releases = {}
    for eid in element_ids:
        releases[str(eid)] = list(release_labels)
    return releases


def build_release_map(release_map):
    """
    Build releases from explicit map.

    Parameters
    ----------
    release_map : dict
        {eid: ["i_rz"], ...}
    """
    releases = {}
    for eid, labels in release_map.items():
        releases[str(eid)] = list(labels)
    return releases


def merge_releases(*release_dicts):
    merged = {}
    for d in release_dicts:
        for eid, labels in d.items():
            if eid not in merged:
                merged[eid] = []
            for label in labels:
                if label not in merged[eid]:
                    merged[eid].append(label)
    return merged


# Central deck frame
release_target_eids = find_elements_by_x_range(
    elements=model_3d["elements"],
    nodes=model_3d["nodes"],
    x_min=430.0,
    x_max=500.0,
    element_types={"3D_frame"},
)

releases = build_element_releases(
    element_ids=release_target_eids[:3],
    release_labels=["i_rz", "j_rz"],
)

# One frame end near Springing
springing_eids = find_elements_by_x_range(
    elements=model_3d["elements"],
    nodes=model_3d["nodes"],
    x_min=120.0,
    x_max=220.0,
    element_types={"3D_frame"},
)

releases = build_element_releases(
    element_ids=springing_eids[:4],
    release_labels=["j_rz"],
)

model = {
    "model_name": "chaotianmen_release_case1",
    "nodes": model_3d["nodes"],
    "elements": model_3d["elements"],
    "supports": model_3d["supports"],
    "nodal_loads": nodal_loads_total,
    "prescribed_displacements": {},
    "member_loads": member_loads_self_weight,
    "temperature_loads": {},
    "fabrication_errors": {},
    "releases": releases,
}

write_model_pretty_compact(
    model,
    "inputs/chaotianmen_release_case1.json",
)